# **Mount Drive**

# **Download Crop Failure Data**

In [ ]:

# Open the text file in read mode ('r')
file_path = '<DATA_ROOT>/crop_failure_links.txt'
link_list = []
with open(file_path, 'r') as file:
  for line in file:
    link_list.append(line)

link_list = sorted(link_list)
link_list = [link[0:-1] for link in link_list]
link_list[-1] = link_list[-1] + 'p'
link_list.pop(0)
link_list

In [ ]:
import requests

def download_file(url,output_file):
  try:
      response = requests.get(url)
      response.raise_for_status()  # Check for any errors during the request

      with open(output_file, 'wb') as file:
          file.write(response.content)

      print(f"File downloaded as '{output_file}'")

  except requests.exceptions.RequestException as e:
      print(f"An error occurred during the request: {e}")
  except Exception as e:
      print(f"An error occurred: {e}")


output_path = '<DATA_ROOT>/crop_failure_data/zipfiles/'

for link in link_list:
  url = link
  output_fname = link.split('/')[-2].replace('-','_') + '.zip'
  output_file = output_path + output_fname
  print(output_file)
  download_file(url,output_file)


In [ ]:
import zipfile
from glob import glob
zip_path = '<DATA_ROOT>/crop_failure_data/zipfiles/*.zip'
files = glob(zip_path)
files

In [ ]:
for zip_file in files:
  extract_dir = '<DATA_ROOT>/crop_failure_data/'
  with zipfile.ZipFile(zip_file, 'r') as zip_ref:
      zip_ref.extractall(extract_dir)


In [ ]:
from glob import glob
fpath = '<DATA_ROOT>/crop_failure_data/*.xlsx'
files = sorted(glob(fpath))
files

In [ ]:
import os
fpath = '<DATA_ROOT>/crop_failure_data/'

for f in files:
  fname = f.split('/')[-1][0:13] + '.xlsx'
  new_file_name = fpath + fname
  old_file_name = f
  os.rename(old_file_name, new_file_name)


# **Preprocess Crop Failure Data**

In [ ]:
import pandas as pd
from glob import glob

fpath = '<DATA_ROOT>/crop_failure_data/crop_failure_raw/*.xlsx'
files = sorted(glob(fpath))
# files = files[3:]
files

In [ ]:
# files[5]

In [ ]:
# df = pd.read_excel(files[8], sheet_name='county_data')
# df

In [ ]:
# df[df['Irrigation Practice']=='O'].Crop.unique()

In [ ]:
# df['Irrigation Practice'].unique()

In [ ]:
# df.Crop.unique()
# df[df.Crop.isin(['SORGHUM','SORGHUM-  DUAL PURPOSE' , 'SORGHUM FORAGE', 'FORAGE SOYBEAN/SORGHUM'])].groupby('Crop').count()

In [ ]:
# cond1 = df['Crop'].str.contains('COTTON', case=False, na=False)
# cond2 = df['Crop'].str.contains('SORGHUM', case=False, na=False)
# cond3 = df['Crop'].str.contains('SOYBEAN', case=False, na=False)
# cond4 = df['Crop'].str.contains('CORN', case=False, na=False)
# cond5 = df['Crop'].str.contains('WHEAT', case=False, na=False)
# cond6 = df['Crop'].str.contains('BARLEY', case=False, na=False)
# cond7 = df['Crop'].str.contains('ALFALFA', case=False, na=False)
# cond8 = df['Crop'].str.contains('OATS', case=False, na=False)

# df[cond1 | cond2 | cond3 | cond4 | cond5 | cond6 | cond7 | cond8]['Crop'].unique()

In [ ]:
import pandas as pd

out_path = '<DATA_ROOT>/crop_failure_data/crop_failure_final/'

cols = [
    'State', 'County','Crop','Crop Type', 'State County Code','Irrigation Practice',
    'Planted Acres','Prevented Acres','Failed Acres']
crops = ['CORN', 'SOYBEANS', 'COTTON', 'WHEAT', 'BARLEY', 'OATS', 'ALFALFA', 'SORGHUM']

state_name_mapping = pd.read_csv('<DATA_ROOT>/state_name_mapping.csv')
state_name_mapping = state_name_mapping.rename(columns={'State1':'State'})

for excel_file in files[3:]:
  df = pd.read_excel(excel_file, sheet_name='county_data')
  df = df[cols]
  cond1 = df['Crop'].str.contains('COTTON', case=False, na=False)
  # cond1 = df['Crop'] == 'COTTON- ELS'
  # cond2 = df['Crop'] == 'COTTON- Upland'
  df.loc[(cond1),'Crop'] = 'COTTON'
  df = df[df['Crop'].isin(crops)]
  if int(excel_file.split('/')[-1][0:4]) < 2017:
    df = pd.merge(df, state_name_mapping, on=['State'], how='inner')
    df = df.drop(columns=['State'])
    df = df.rename(columns={'State2':'State'})
    df = df[cols]
  df = df.rename(columns={'State County Code':'FIPS'})
  fname = excel_file.split('/')[-1]
  print(fname)
  df.to_excel(out_path+fname)


In [ ]:
df.groupby('Crop').count()

In [ ]:
import pandas as pd
from glob import glob
fpath = '<DATA_ROOT>/crop_failure_data/crop_failure_final/*.xlsx'
files = sorted(glob(fpath))
files

In [ ]:
df = pd.read_excel(files[2])


In [ ]:
df.groupby(['Irrigation Practice']).count()

# New Section

In [ ]:
# 2011
# 'Planted Acres','Prevented Acres','Planted and Failed Acres'
# 'Planted\nTotal' , 'Prevented\nTotal' , 'Plant and Fail\nTotal',
# 'Planted\nIrrigated', 'Prevented\nIrrigated','Plant and Fail\nIrrigated'
# 'Planted\nNonirrigated', 'Prevented\nNonirrigated','Plant and Fail\nNonirrigated'

In [ ]:
fpath = '<DATA_ROOT>/crop_failure_data/crop_failure_raw/*.xlsx'
files = sorted(glob(fpath))
files

In [ ]:
files[2]

'<DATA_ROOT>/crop_failure_data/crop_failure_raw/2011_fsa_acre.xlsx'

In [ ]:
df = pd.read_excel(files[2], sheet_name='Sheet1')
df

In [ ]:
df['FIPS'] = df['State Code'] * 1000 + df['County Code']
cols0 = list(df.columns)[0:9] + ['FIPS']
cols_all = cols0 + list(df.columns)[9:12]
cols_ig = cols0 + list(df.columns)[15:18]
cols_ng = cols0 + list(df.columns)[21:24]
df_all = df.loc[:,cols_all]
df_ig = df.loc[:,cols_ig]
df_ng = df.loc[:,cols_ng]
df_all = df_all.drop(columns=['State Code','County Code','Crop Code','Crop Type Name'])
df_ig = df_ig.drop(columns=['State Code','County Code','Crop Code','Crop Type Name'])
df_ng = df_ng.drop(columns=['State Code','County Code','Crop Code','Crop Type Name'])

rename_cols_all = {
    'State Name' : 'State' , 'County Name' : 'County' , 'Crop Name' : 'Crop' , 'Crop Type' : 'Crop Type' ,
    'Planted\nTotal' : 'Planted Acres' , 'Prevented\nTotal' : 'Prevented Acres' , 'Failed\nTotal' : 'Failed Acres'
}
df_all = df_all.rename(columns = rename_cols_all)

rename_cols_ig = {
    'State Name' : 'State' , 'County Name' : 'County' , 'Crop Name' : 'Crop' , 'Crop Type' : 'Crop Type' ,
    'Planted\nIrrigated' : 'Planted Acres' , 'Prevented\nIrrigated' : 'Prevented Acres' , 'Failed\nIrrigated' : 'Failed Acres'
}
df_ig = df_ig.rename(columns = rename_cols_ig)

rename_cols_ng = {
    'State Name' : 'State' , 'County Name' : 'County' , 'Crop Name' : 'Crop' , 'Crop Type' : 'Crop Type' ,
    'Planted\nNonirrigated' : 'Planted Acres' , 'Prevented\nNonirrigated' : 'Prevented Acres' , 'Failed\nNonirrigated' : 'Failed Acres'
}
df_ng = df_ng.rename(columns = rename_cols_ng)

df_all['Irrigation Practice'] = 'ALL'
df_ig['Irrigation Practice'] = 'I'
df_ng['Irrigation Practice'] = 'N'

df_2011 = pd.concat([df_ig,df_ng,df_all],ignore_index=True)

In [ ]:
df_2011#[df_2011['Failed Acres']>0].groupby('Irrigation Practice').count()

In [ ]:
crops = ['CORN', 'SOYBEANS', 'COTTON', 'WHEAT', 'BARLEY', 'OATS', 'ALFALFA', 'SORGHUM']
cond1 = df_2011['Crop'].str.contains('COTTON', case=False, na=False)
# cond1 = df_2011['Crop'] == 'COTTON- ELS'
# cond2 = df_2011['Crop'] == 'COTTON- Upland'
df_2011.loc[(cond1),'Crop'] = 'COTTON'
df_2011 = df_2011[df_2011['Crop'].isin(crops)]

# cond1 = df_2011['Crop'] == 'ALFALFA'
# df_2011.loc[(cond1),'Crop'] = 'HAY'

state_name_mapping = pd.read_csv('<DATA_ROOT>/state_name_mapping.csv')
state_name_mapping = state_name_mapping.rename(columns={'State1':'State'})

df_2011 = pd.merge(df_2011, state_name_mapping, on=['State'], how='inner')
df_2011 = df_2011.drop(columns=['State'])
df_2011 = df_2011.rename(columns={'State2':'State'})

cols = [
    'State', 'County','Crop','Crop Type', 'FIPS','Irrigation Practice',
    'Planted Acres','Prevented Acres','Failed Acres']

df_2011 = df_2011[cols]

df_2011

In [ ]:
df_2011.Crop.unique()

array(['WHEAT', 'OATS', 'COTTON', 'CORN', 'SORGHUM', 'SOYBEANS',
       'ALFALFA', 'BARLEY'], dtype=object)

In [ ]:
out_path = '<DATA_ROOT>/crop_failure_data/crop_failure_final/'
fname = files[2].split('/')[-1]
print(fname)
df_2011.to_excel(out_path+fname)

2011_fsa_acre.xlsx


# New Section

In [ ]:
df0 = pd.read_excel(files[2], sheet_name='Sheet1')
df0 = df0.loc[:,['State Code' , 'State Name']]
df_st_codes = df0.groupby('State Name').first().reset_index()
df_st_codes

In [ ]:
df_2010 = pd.read_excel(files[1], sheet_name='Sheet1')
df_2010

In [ ]:
df_2010

In [ ]:
# df_2010 = pd.read_excel(files[1], sheet_name='Sheet1')
df_2010['FIPS'] = df_2010['State'] * 1000 + df_2010['County Code']
df_2010 = df_2010.rename(columns={'State':'State Code'})
df_2010 = pd.merge(df_2010,df_st_codes, on=['State Code'], how='inner')
df_2010 = df_2010.drop(columns=['State Code','County Code','State Abbrev','Crop Code','Crop Type Name'])

rename_cols_2010 = {
    'State Name' : 'State' , 'County Name' : 'County' , 'Crop Name' : 'Crop' , 'Crop Type' : 'Crop Type'
}
df_2010 = df_2010.rename(columns = rename_cols_2010)

cols = [
    'State', 'County','Crop','Crop Type', 'FIPS','Irrigation Practice',
    'Planted Acres','Prevented Acres','Failed Acres']
df_2010['Irrigation Practice'] = 'ALL'

state_name_mapping = pd.read_csv('<DATA_ROOT>/state_name_mapping.csv')
state_name_mapping = state_name_mapping.rename(columns={'State1':'State'})

df_2010 = pd.merge(df_2010, state_name_mapping, on=['State'], how='inner')
df_2010 = df_2010.drop(columns=['State'])
df_2010 = df_2010.rename(columns={'State2':'State'})
df_2010 = df_2010[cols]

crops = ['CORN', 'SOYBEANS', 'COTTON', 'WHEAT', 'BARLEY', 'OATS', 'ALFALFA', 'SORGHUM']
cond1 = df_2010['Crop'].str.contains('COTTON', case=False, na=False)
# cond1 = df_2010['Crop'] == 'COTTON- ELS'
# cond2 = df_2010['Crop'] == 'COTTON- Upland'
df_2010.loc[(cond1),'Crop'] = 'COTTON'
df_2010 = df_2010[df_2010['Crop'].isin(crops)]

# cond1 = df_2010['Crop'] == 'ALFALFA'
# df_2010.loc[(cond1),'Crop'] = 'HAY'

df_2010

out_path = '<DATA_ROOT>/crop_failure_data/crop_failure_final/'
fname = files[1].split('/')[-1]
print(fname)
df_2010.to_excel(out_path+fname)


2010_fsa_acre.xlsx


In [ ]:
df_st_codes = df_st_codes.rename(columns={'State Name':'State Name0'})
df_st_codes

In [ ]:
df_2009 = pd.read_excel(files[0], sheet_name='Sheet1')
df_2009

In [ ]:
df_2009

In [ ]:
df_2009['FIPS'] = df_2009['State Code'] * 1000 + df_2009['County Code']

# df_2010 = df_2010.rename(columns={'State':'State Code'})
df_2009 = pd.merge(df_2009,df_st_codes, on=['State Code'], how='inner')
df_2009 = df_2009.drop(columns=['State Code','County Code','State Name','Crop Code','Crop Type Name'])
rename_cols_2009 = {
    'State Name0' : 'State' , 'County Name' : 'County' , 'Crop Name' : 'Crop' , 'Crop Type' : 'Crop Type'
}
df_2009 = df_2009.rename(columns = rename_cols_2009)

cols = [
    'State', 'County','Crop','Crop Type', 'FIPS','Irrigation Practice',
    'Planted Acres','Prevented Acres','Failed Acres']
df_2009['Irrigation Practice'] = 'ALL'

state_name_mapping = pd.read_csv('<DATA_ROOT>/state_name_mapping.csv')
state_name_mapping = state_name_mapping.rename(columns={'State1':'State'})

df_2009 = pd.merge(df_2009, state_name_mapping, on=['State'], how='inner')
df_2009 = df_2009.drop(columns=['State'])
df_2009 = df_2009.rename(columns={'State2':'State'})
df_2009 = df_2009[cols]

crops = ['CORN', 'SOYBEANS', 'COTTON', 'WHEAT', 'BARLEY', 'OATS', 'ALFALFA', 'SORGHUM']
cond1 = df_2009['Crop'].str.contains('COTTON', case=False, na=False)
# cond1 = df_2009['Crop'] == 'COTTON- ELS'
# cond2 = df_2009['Crop'] == 'COTTON- Upland'
df_2009.loc[(cond1),'Crop'] = 'COTTON'
df_2009 = df_2009[df_2009['Crop'].isin(crops)]

# cond1 = df_2009['Crop'] == 'ALFALFA'
# df_2009.loc[(cond1),'Crop'] = 'HAY'

In [ ]:
df_2009

In [ ]:

out_path = '<DATA_ROOT>/crop_failure_data/crop_failure_final/'
fname = files[0].split('/')[-1]
print(fname)
df_2009.to_excel(out_path+fname)


2009_fsa_acre.xlsx
